# Tahap 1: Membangun Case Base
**CBR - Pidana Umum: Pemalsuan**

Tujuan: Mengumpulkan dan menyiapkan corpus putusan yang bersih dari file PDF.

**Langkah:**
1. Konversi PDF → plain text
2. Pembersihan teks (hapus header/footer, normalisasi)
3. Validasi kelengkapan teks
4. Simpan ke `/data/raw/*.txt`

## 0. Instalasi Library

In [1]:
# Jalankan sekali saja untuk instalasi
!pip install pdfminer.six tqdm


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Import Library & Konfigurasi Path

In [8]:
import os
import re
import json
import logging
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from pdfminer.high_level import extract_text
from pdfminer.pdfparser import PDFSyntaxError

# ============================================================
# KONFIGURASI - Sesuaikan path folder PDF Anda di sini
# ============================================================
PDF_INPUT_DIR  = Path("../data/pdf_input")        # Folder tempat 35 PDF Anda berada
RAW_OUTPUT_DIR = Path("../data/raw")         # Output teks bersih
LOG_DIR        = Path("../logs")
MIN_CONTENT_RATIO = 0.80                     # Minimal 80% isi putusan harus ada
MIN_WORD_COUNT    = 200                      # Minimum kata agar dianggap valid
# ============================================================

# Buat direktori jika belum ada
for d in [PDF_INPUT_DIR, RAW_OUTPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Setup logging
log_file = LOG_DIR / "cleaning.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info("=== Tahap 1: Membangun Case Base - Mulai ===")
print(f"✅ Konfigurasi selesai. Log disimpan di: {log_file}")

2026-06-15 18:07:52,787 | INFO | === Tahap 1: Membangun Case Base - Mulai ===


✅ Konfigurasi selesai. Log disimpan di: ..\logs\cleaning.log


## 2. Konversi PDF → Plain Text

In [9]:
def pdf_to_text(pdf_path: Path) -> str:
    """
    Konversi file PDF ke plain text menggunakan pdfminer.
    Mengembalikan string teks, atau string kosong jika gagal.
    """
    try:
        text = extract_text(str(pdf_path))
        return text if text else ""
    except PDFSyntaxError as e:
        logger.error(f"PDFSyntaxError pada {pdf_path.name}: {e}")
        return ""
    except Exception as e:
        logger.error(f"Error membaca {pdf_path.name}: {e}")
        return ""

# Cek file PDF yang tersedia
pdf_files = sorted(PDF_INPUT_DIR.glob("*.pdf"))
print(f"📄 Ditemukan {len(pdf_files)} file PDF di folder: {PDF_INPUT_DIR}")
if len(pdf_files) == 0:
    print("⚠️  PERHATIAN: Tidak ada PDF ditemukan!")
    print(f"   Pastikan file PDF ada di folder: {PDF_INPUT_DIR.resolve()}")
else:
    for i, f in enumerate(pdf_files[:5]):
        print(f"  [{i+1}] {f.name}")
    if len(pdf_files) > 5:
        print(f"  ... dan {len(pdf_files)-5} file lainnya")

📄 Ditemukan 35 file PDF di folder: ..\data\pdf_input
  [1] putusan_1014_k__pid__2014_20260614213825.pdf
  [2] putusan_1122_k_pid_2015_20260614211456.pdf
  [3] putusan_1152_k_pid_2016_20260614212649.pdf
  [4] putusan_1382_k_pid_2016_20260614213455.pdf
  [5] putusan_1394_k_pid_2013_20260614212706.pdf
  ... dan 30 file lainnya


## 3. Fungsi Pembersihan Teks

In [10]:
# Pola header/footer putusan MA RI yang umum
HEADER_FOOTER_PATTERNS = [
    r'Mahkamah Agung Republik Indonesia',
    r'Direktori Putusan',
    r'putusan\.mahkamahagung\.go\.id',
    r'Disclaimer[\s\S]{0,200}?digunakan',
    r'Halaman \d+ dari \d+',
    r'hal\.\s*\d+\s*dari\s*\d+',
    r'-{3,}',          # Garis pemisah
    r'\f',             # Form feed / page break
]

def clean_text(raw_text: str) -> str:
    """
    Membersihkan teks putusan:
    1. Hapus header/footer MA RI
    2. Hapus nomor halaman
    3. Normalisasi spasi dan karakter
    4. Lower-case
    """
    text = raw_text
    
    # 1. Hapus watermark & header/footer
    for pattern in HEADER_FOOTER_PATTERNS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)
    
    # 2. Hapus nomor halaman standalone
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    
    # 3. Normalisasi karakter khusus
    text = text.replace('\u2019', "'").replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\xa0', ' ')   # Non-breaking space
    text = text.replace('\t', ' ')     # Tab
    
    # 4. Hapus karakter non-ASCII berlebih (pertahankan huruf Indo)
    text = re.sub(r'[^\w\s.,;:\-()\[\]/"\'\']+', ' ', text)
    
    # 5. Normalisasi spasi berganda dan baris kosong berlebih
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # 6. Lower-case
    text = text.lower()
    
    # 7. Strip awal-akhir
    text = text.strip()
    
    return text

print("✅ Fungsi clean_text siap digunakan")

✅ Fungsi clean_text siap digunakan


## 4. Fungsi Validasi

In [11]:
# Kata kunci wajib yang harus ada di putusan pidana pemalsuan
REQUIRED_KEYWORDS = [
    'terdakwa', 'dakwaan', 'tuntutan', 'putusan', 'majelis hakim'
]

def validate_text(text: str, filename: str) -> dict:
    """
    Validasi kelengkapan teks putusan.
    Mengembalikan dict dengan status dan detail validasi.
    """
    word_count = len(text.split())
    char_count = len(text)
    
    # Cek kata kunci wajib
    found_keywords = [kw for kw in REQUIRED_KEYWORDS if kw in text.lower()]
    keyword_ratio = len(found_keywords) / len(REQUIRED_KEYWORDS)
    
    # Estimasi kelengkapan (berdasarkan jumlah kata vs minimum)
    content_ok = word_count >= MIN_WORD_COUNT
    keyword_ok = keyword_ratio >= MIN_CONTENT_RATIO
    
    is_valid = content_ok and keyword_ok
    
    result = {
        "filename": filename,
        "word_count": word_count,
        "char_count": char_count,
        "keywords_found": found_keywords,
        "keyword_ratio": round(keyword_ratio, 2),
        "is_valid": is_valid,
        "status": "VALID" if is_valid else "PERLU CEK"
    }
    return result

print("✅ Fungsi validasi siap")

✅ Fungsi validasi siap


## 5. Pipeline Utama: PDF → Clean Text

In [12]:
validation_results = []
success_count = 0
fail_count = 0

logger.info(f"Memproses {len(pdf_files)} file PDF...")

for i, pdf_path in enumerate(tqdm(pdf_files, desc="Memproses PDF")):
    case_id = f"case_{i+1:03d}"
    output_path = RAW_OUTPUT_DIR / f"{case_id}.txt"
    
    # Step 1: Konversi PDF ke teks
    raw_text = pdf_to_text(pdf_path)
    
    if not raw_text.strip():
        logger.warning(f"[{case_id}] {pdf_path.name} - Teks kosong, skip")
        fail_count += 1
        validation_results.append({
            "case_id": case_id,
            "original_filename": pdf_path.name,
            "filename": f"{case_id}.txt",
            "word_count": 0,
            "char_count": 0,
            "keywords_found": [],
            "keyword_ratio": 0,
            "is_valid": False,
            "status": "GAGAL - Teks Kosong"
        })
        continue
    
    # Step 2: Bersihkan teks
    clean = clean_text(raw_text)
    
    # Step 3: Validasi
    val = validate_text(clean, f"{case_id}.txt")
    val["case_id"] = case_id
    val["original_filename"] = pdf_path.name
    validation_results.append(val)
    
    # Step 4: Simpan
    output_path.write_text(clean, encoding="utf-8")
    
    if val["is_valid"]:
        success_count += 1
        logger.info(f"[{case_id}] {pdf_path.name} → {val['word_count']} kata | {val['status']}")
    else:
        fail_count += 1
        logger.warning(f"[{case_id}] {pdf_path.name} → {val['word_count']} kata | {val['status']}")

# Simpan mapping case_id → filename
mapping = [{"case_id": v["case_id"], "original_filename": v["original_filename"]} for v in validation_results]
with open(RAW_OUTPUT_DIR / "case_mapping.json", "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print(f"\n{'='*50}")
print(f"✅ Berhasil diproses : {success_count} dokumen")
print(f"⚠️  Perlu dicek      : {fail_count} dokumen")
print(f"📁 Output tersimpan di: {RAW_OUTPUT_DIR.resolve()}")

2026-06-15 18:08:06,842 | INFO | Memproses 35 file PDF...
Memproses PDF: 100%|██████████| 35/35 [01:20<00:00,  2.31s/it]


✅ Berhasil diproses : 35 dokumen
⚠️  Perlu dicek      : 0 dokumen
📁 Output tersimpan di: E:\Semester 6\Penalaran Komputer\SubCPMK 3\Test_1\data\raw


## 6. Laporan Validasi

In [13]:
import pandas as pd

df_val = pd.DataFrame(validation_results)

print("📊 Ringkasan Validasi Dokumen:")
print(f"  Total dokumen       : {len(df_val)}")
print(f"  Valid               : {df_val['is_valid'].sum()}")
print(f"  Perlu dicek         : {(~df_val['is_valid']).sum()}")
print(f"  Rata-rata kata      : {df_val['word_count'].mean():.0f} kata")
print(f"  Min kata            : {df_val['word_count'].min()} kata")
print(f"  Max kata            : {df_val['word_count'].max()} kata")
print()

# Tampilkan detail
cols = ["case_id", "original_filename", "word_count", "keyword_ratio", "status"]
print(df_val[cols].to_string(index=False))

# Simpan laporan
df_val.to_csv(LOG_DIR / "validation_report.csv", index=False, encoding="utf-8-sig")
print(f"\n📄 Laporan validasi disimpan: {LOG_DIR}/validation_report.csv")

📊 Ringkasan Validasi Dokumen:
  Total dokumen       : 35
  Valid               : 35
  Perlu dicek         : 0
  Rata-rata kata      : 8662 kata
  Min kata            : 1934 kata
  Max kata            : 83980 kata

 case_id                            original_filename  word_count  keyword_ratio status
case_001 putusan_1014_k__pid__2014_20260614213825.pdf        5815            1.0  VALID
case_002   putusan_1122_k_pid_2015_20260614211456.pdf       13756            1.0  VALID
case_003   putusan_1152_k_pid_2016_20260614212649.pdf        6903            1.0  VALID
case_004   putusan_1382_k_pid_2016_20260614213455.pdf        4132            1.0  VALID
case_005   putusan_1394_k_pid_2013_20260614212706.pdf        6194            1.0  VALID
case_006     putusan_13_k_pid_2017_20260614212521.pdf       20735            1.0  VALID
case_007   putusan_1530_k_pid_2015_20260614212713.pdf        6089            1.0  VALID
case_008   putusan_1543_k_pid_2012_20260614212255.pdf        4814            1.0  

## 7. Preview Contoh Teks Bersih

In [14]:
# Tampilkan preview 500 karakter pertama dari case pertama yang valid
txt_files = sorted(RAW_OUTPUT_DIR.glob("*.txt"))
if txt_files:
    sample_file = txt_files[0]
    sample_text = sample_file.read_text(encoding="utf-8")
    print(f"📄 Preview: {sample_file.name}")
    print("-" * 60)
    print(sample_text[:800])
    print("-" * 60)
    print(f"Total kata: {len(sample_text.split())}")
else:
    print("⚠️ Tidak ada file teks. Pastikan folder PDF sudah benar.")

print("\n✅ Tahap 1 selesai! Lanjutkan ke notebook 02_case_representation.ipynb")

📄 Preview: case_001.txt
------------------------------------------------------------
disclaimer
kepaniteraan berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen mahkamah agung untuk pelayanan publik, transparansi dan akuntabilitas
pelaksanaan fungsi peradilan. namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.
dalam hal anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi kepaniteraan mahkamah agung ri melalui :
email : kepaniteraan mahkamahagung.go.id telp : 021-384 3348 (ext.318)

halaman 1

 hal. put. nomor 1014 k/pid/2014p u t u s a n nomor 1014 k /pid/ 2014 demi keadilan berdasarkan ketuhanan
------------------------------------------------------------
Total kata: 5815

✅ Tahap 1 selesai! Lanjutkan ke not